In [1]:
1+1

2

In [2]:
import os
from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI

C:\Users\Admin\AppData\Local\Temp\ipykernel_8000\4142441295.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [5]:
from langchain_community.llms import Ollama

# Connect to your local Llama 3.2 model
llm = Ollama(model="llama3.2")

response = llm.invoke("Explain what LangSmith does in simple terms.")
print(response)

LangSmith is a tool that helps people understand the meaning of words and phrases in context. It's like a thesaurus that suggests alternative words or phrases that can help you express yourself more clearly and accurately.

Imagine you're writing a story or article, and you come across a word that you're not sure how to define or use in the right way. LangSmith provides options for you to choose from, so you can select the best word to fit your meaning.


In [6]:
loader = TextLoader("Knowledge.txt")

In [7]:
document = loader.load()

In [8]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
docs = splitter.split_documents(document)

In [9]:
docs

[Document(metadata={'source': 'Knowledge.txt'}, page_content='Johnathan "John" Miles is a man defined by a meticulous and aspirational "love" for James Bond, a passion that borders on lifestyle emulation. By day, John is a 44-year-old risk analyst in Chicago, a profession steeped in data and predictability, which he actively contrasts with the sophisticated, crisis-driven world of 007. This fixation is not just casual fandom; it is a dedicated curation of his personal aesthetic and environment. He invests significantly in the specific trappings of the character, favoring navy blue, single-breasted suits, brown suede chukka boots, and a specific Omega Seamaster Diver 300M, which he acquired after his last bonus. His evenings are often spent perfecting the Vesper martiniâ€”having researched the exact non-mainstream ingredients like Lillet Blancâ€”or streaming the John Barry instrumental scores. For John, the "love" for Bond is an appreciation for an ideal of precision, classic masculinit

In [10]:
embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

c:\Users\Admin\Desktop\Projects\BASIC_RAG_1\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1356.05it/s]


In [11]:
vectorstore = Chroma.from_documents(docs,embedding=embedding,persist_directory="./chroma_db")

In [12]:
retriever = vectorstore.as_retriever(search_kwargs={"k":3})

In [13]:
prompt = ChatPromptTemplate.from_template("""
You are an AI assistant. Use the following content to answer the question.
                                        Context: {context}
                                        Question:{question}
                                          give a concise, factual answer.

""")

In [14]:
rag_chain=(
    {
        "context":retriever | (lambda docs:"\n\n".join([d.page_content for d in docs])),
        "question":RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()

)

In [ ]:
#testing 1 and output of the model

In [ ]:
query = rag_chain.invoke("who is jhon")    

In [ ]:
query                                      

'Johnathan "John" Miles is a 44-year-old risk analyst in Chicago.'

In [24]:
#testing 2 and output of the model

In [32]:
query = rag_chain.invoke("Jhon is the fan of which fictional character")


In [33]:
query

'James Bond.'